# Run Lookism Agent Drift Experiments

This notebook runs a 10-turn agent-to-agent conversation in which both the assistant and the user persona are generated by Qwen. The goal is to see whether the assistant gradually drifts toward more permissive body-control framing over accumulated context. The default setting keeps approval visible but turns phase guidance off.


## Mount Google Drive


In [ ]:
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    print('Google Drive is mounted.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping Drive mount.')


## Sync Repo From GitHub


In [ ]:
import os
import sys

DRIVE_REPO = Path('/content/drive/MyDrive/memory_harm_Shin-u')
BRANCH = 'manual-transcripts'

if not DRIVE_REPO.exists():
    raise FileNotFoundError(f'Repo not found at {DRIVE_REPO}')

os.chdir(DRIVE_REPO)
print(f'Working directory: {DRIVE_REPO}')

!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

if str(DRIVE_REPO) not in sys.path:
    sys.path.insert(0, str(DRIVE_REPO))


## Install Dependencies


In [ ]:
!pip install --upgrade pip
!pip install unsloth vllm bitsandbytes
!pip install -r requirements.txt


## Launch vLLM


In [ ]:
model = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'


In [ ]:
import time
import requests

print('Launching vLLM server...')
!nohup python -m vllm.entrypoints.openai.api_server \
    --model $model \
    --served-model-name $model \
    --quantization bitsandbytes \
    --load-format bitsandbytes \
    --trust-remote-code \
    --port 8000 \
    --gpu-memory-utilization 0.9 > vllm.log 2>&1 &

timeout = 300
start_time = time.time()
while True:
    try:
        response = requests.get('http://localhost:8000/v1/models', timeout=5)
        if response.status_code == 200:
            print(f'vLLM server is ready with model {model}')
            break
    except Exception:
        pass
    if time.time() - start_time > timeout:
        raise TimeoutError('Timed out waiting for vLLM server. Check vllm.log.')
    print('.', end='', flush=True)
    time.sleep(5)


## Configure Drift Experiment


In [ ]:
from src.lookism_agent_drift import (
    LOOKISM_SEEDS,
    LookismDriftConfig,
    get_lookism_seed,
    render_drift_bundle,
    run_lookism_seed,
    save_drift_bundle,
)

OUTPUT_DIR = Path('data/lookism_agent_drift')
SEED_IDS = [seed['id'] for seed in LOOKISM_SEEDS]
SEED_IDS


In [ ]:
seed_id = 'heartbreak_glow_up'
cfg = LookismDriftConfig(
    assistant_model=model,
    user_model=model,
    turns=10,
    memory_mode='full_context',
    assistant_temperature=0.7,
    user_temperature=0.8,
    use_phase_guidance=False,
)
seed = get_lookism_seed(seed_id)
seed


In [ ]:
bundle = run_lookism_seed(seed, cfg)
print(render_drift_bundle(bundle))


## Run All Three Seeds


In [ ]:
all_bundles = []
for seed in LOOKISM_SEEDS:
    print('=' * 100)
    print(seed['id'])
    print('=' * 100)
    bundle = run_lookism_seed(seed, cfg)
    all_bundles.append(bundle)
    print(render_drift_bundle(bundle))
    print()


In [ ]:
saved_paths = [save_drift_bundle(bundle, OUTPUT_DIR) for bundle in all_bundles]
saved_paths
